<style>
*{
    direction: rtl;
    text-align: right;
}
</style>

# تمرین CNN — «آزمایشگاه حال‌سنجی ایموجی‌ها»
## نسخه‌ی تمرین برای دانشجو

در این پروژه قرار است یک **CNN از صفر** برای تشخیص حالت چهره‌ی ایموجی‌ها بسازیم.  
ایده‌ی پروژه این است که یک ربات بامزه باید از روی تصویر ایموجی‌ها تشخیص بدهد که حال آن‌ها چیست:

- `happy`
- `sad`
- `angry`
- `surprised`
- `sleepy`

## داستان پروژه
در شهر ایموجی‌ها، یک ربات کوچک وظیفه دارد پیام‌های تصویری را مرتب‌سازی کند.  
اما ایموجی‌ها همیشه تمیز و واضح نیستند: گاهی کمی چرخیده‌اند، گاهی نویز دارند، گاهی جابه‌جا شده‌اند.  
هدف شما این است که یک مدل CNN طراحی کنید که در چنین شرایطی هم بتواند حالت چهره را درست تشخیص دهد.

## بعد از انجام این نوت‌بوک باید بتوانید:
1. یک دیتاست تصویری را بسازید یا بارگذاری کنید.
2. روی داده‌ها **EDA** ساده انجام دهید.
3. **Data Augmentation** مناسب برای مسئله‌ی تصویری انتخاب کنید.
4. یک **CNN پایه** را از صفر پیاده‌سازی و آموزش دهید.
5. با استفاده از **loss curve**، **accuracy**، **macro F1** و **confusion matrix** مدل را تحلیل کنید.
6. خطاهای مدل را بررسی کنید و یک نسخه‌ی بهبود‌یافته از مدل ارائه دهید.

<style>
*{
    direction: rtl;
    text-align: right;
}
</style>

## بخش ۰ — کتابخانه‌ها و تنظیمات اولیه

### صورت سؤال
در این بخش:
1. کتابخانه‌های موردنیاز را import کنید.
2. seed را برای بازتولیدپذیری تنظیم کنید.
3. دستگاه اجرا (`cpu` یا `cuda`) را مشخص کنید.
4. تنظیمات کلی پروژه مثل اندازه‌ی تصویر، batch size و نام کلاس‌ها را تعریف کنید.

### خروجی مورد انتظار
- چاپ دستگاه اجرا
- چاپ نام کلاس‌ها

In [1]:
import math
import random
from copy import deepcopy

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from PIL import Image, ImageDraw, ImageFilter, ImageOps, ImageEnhance

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

IMG_SIZE = 64
BATCH_SIZE = 64
NUM_CLASSES = 5
CLASS_NAMES = ['happy', 'sad', 'angry', 'surprised', 'sleepy']

print('Device:', device)
print('Classes:', CLASS_NAMES)

KeyboardInterrupt: 

<style>
*{
    direction: rtl;
    text-align: right;
}
</style>

## بخش ۱ — ساخت دیتاست مصنوعی ایموجی‌ها

### صورت سؤال
به‌جای دانلود دیتاست آماده، در این پروژه یک دیتاست کوچک و بامزه را **خودمان تولید می‌کنیم**.

1. یک تابع بنویسید که برای هر کلاس، یک تصویر ایموجی تولید کند.
2. در تولید هر تصویر کمی تصادفی‌سازی داشته باشید؛ مثلاً:
   - جابه‌جایی مرکز صورت
   - تغییر شعاع صورت
   - تغییر جزئی در اجزای چهره
   - نویز ساده‌ی تصویری
3. تفاوت حالت‌های چهره باید از طریق **چشم‌ها، ابروها و دهان** مشخص باشد.

### خروجی مورد انتظار
- توابع تولید تصویر و نویز

In [ ]:
# Synthetic emoji generator -------------------------------------------------
# TODO: complete the image-generation logic in the functions below.

def _jitter(rng, low, high):
    return rng.randint(low, high)

def add_soft_noise(img, rng, amount=35):
    """Add light Gaussian noise to a PIL image.

    TODO:
    - create a noise array
    - add it to the image
    - clip the result to a valid image range
    - return a PIL image
    """
    arr = np.array(img).astype(np.int16)
    # TODO: generate noise and blend it with the image array
    # TODO: clip the values to [0, 255]
    raise NotImplementedError('TODO: implement add_soft_noise')

def draw_emoji(label, img_size=64, seed=0, hard_mode=False):
    """Generate a synthetic emoji image for one emotion class.

    TODO:
    - create a background canvas
    - draw the face circle
    - encode the class using eyes / eyebrows / mouth
    - optionally add rotation, blur, and noise
    """
    rng = random.Random(seed)

    bg_color = (
        245 + _jitter(rng, -8, 8),
        245 + _jitter(rng, -8, 8),
        250 + _jitter(rng, -8, 5),
    )
    img = Image.new('RGB', (img_size, img_size), bg_color)
    draw = ImageDraw.Draw(img)

    # TODO: choose a face center and radius
    cx = img_size // 2 + _jitter(rng, -3, 3)
    cy = img_size // 2 + _jitter(rng, -3, 3)
    radius = 22 + _jitter(rng, -2, 2)

    # TODO: draw the face and class-specific features
    # Suggested features: eyes, eyebrows, mouth, cheeks, tear / sleep bubble
    # Suggested augmentations: random rotation, mild blur, soft noise, light occlusion
    raise NotImplementedError('TODO: implement draw_emoji')

<style>
*{
    direction: rtl;
    text-align: right;
}
</style>

## بخش ۲ — مشاهده‌ی نمونه‌ها و بررسی اولیه‌ی داده

### صورت سؤال
1. از هر کلاس چند نمونه تصویر نمایش دهید.
2. درباره‌ی تفاوت‌های ظاهری کلاس‌ها توضیح کوتاه بنویسید.
3. تعداد نمونه‌های هر کلاس در هر split را مشخص کنید.
4. یک نمودار میله‌ای از توزیع داده‌ها رسم کنید.

### خروجی مورد انتظار
- یک grid از نمونه‌تصاویر
- جدول یا نمودار تعداد نمونه‌ها

In [ ]:
# TODO [03]: Complete the code for بخش ۲ — مشاهده‌ی نمونه‌ها و بررسی اولیه‌ی داده.
# The instructions for this task are described in the markdown cell above.
# Replace this placeholder with your implementation.

In [ ]:
# TODO [04]: Complete the code for بخش ۲ — مشاهده‌ی نمونه‌ها و بررسی اولیه‌ی داده.
# The instructions for this task are described in the markdown cell above.
# Replace this placeholder with your implementation.

<style>
*{
    direction: rtl;
    text-align: right;
}
</style>

## بخش ۳ — پیش‌پردازش و Data Augmentation

### صورت سؤال
1. برای داده‌های آموزشی یک pipeline مناسب از augmentation تعریف کنید.
2. برای داده‌های اعتبارسنجی و تست فقط پیش‌پردازش‌های ضروری را اعمال کنید.
3. توضیح دهید چرا در این مسئله این تبدیلات مناسب‌اند.

### پیشنهاد
برای train می‌توانید از موارد زیر استفاده کنید:
- `RandomRotation`
- `RandomResizedCrop`
- `ColorJitter`
- `RandomHorizontalFlip`

### خروجی مورد انتظار
- تعریف transformهای train / val / test
- نمایش یک نمونه تصویر قبل و بعد از augmentation

In [ ]:
# These custom transforms are provided as helpers.
# TODO: choose the final train / validation / test pipelines below.

class Compose:
    def __init__(self, transforms):
        self.transforms = transforms

    def __call__(self, img):
        for t in self.transforms:
            img = t(img)
        return img

class Resize:
    def __init__(self, size):
        if isinstance(size, int):
            size = (size, size)
        self.size = size

    def __call__(self, img):
        return img.resize(self.size, resample=Image.BILINEAR)

class RandomHorizontalFlip:
    def __init__(self, p=0.5):
        self.p = p

    def __call__(self, img):
        if random.random() < self.p:
            return ImageOps.mirror(img)
        return img

class RandomRotation:
    def __init__(self, degrees):
        self.degrees = degrees

    def __call__(self, img):
        angle = random.uniform(-self.degrees, self.degrees)
        bg = img.getpixel((0, 0))
        return img.rotate(angle, resample=Image.BILINEAR, fillcolor=bg)

class RandomResizedCrop:
    def __init__(self, size, scale=(0.85, 1.0)):
        self.size = (size, size) if isinstance(size, int) else size
        self.scale = scale

    def __call__(self, img):
        w, h = img.size
        factor = random.uniform(self.scale[0], self.scale[1])
        crop_w = max(8, int(w * factor))
        crop_h = max(8, int(h * factor))
        left = random.randint(0, max(0, w - crop_w))
        top = random.randint(0, max(0, h - crop_h))
        img = img.crop((left, top, left + crop_w, top + crop_h))
        return img.resize(self.size, resample=Image.BILINEAR)

class ColorJitter:
    def __init__(self, brightness=0.0, contrast=0.0, saturation=0.0):
        self.brightness = brightness
        self.contrast = contrast
        self.saturation = saturation

    def __call__(self, img):
        if self.brightness > 0:
            factor = random.uniform(1 - self.brightness, 1 + self.brightness)
            img = ImageEnhance.Brightness(img).enhance(factor)
        if self.contrast > 0:
            factor = random.uniform(1 - self.contrast, 1 + self.contrast)
            img = ImageEnhance.Contrast(img).enhance(factor)
        if self.saturation > 0:
            factor = random.uniform(1 - self.saturation, 1 + self.saturation)
            img = ImageEnhance.Color(img).enhance(factor)
        return img

class ToTensorNormalize:
    def __init__(self, mean, std):
        self.mean = np.array(mean, dtype=np.float32).reshape(3, 1, 1)
        self.std = np.array(std, dtype=np.float32).reshape(3, 1, 1)

    def __call__(self, img):
        arr = np.asarray(img).astype(np.float32) / 255.0
        arr = np.transpose(arr, (2, 0, 1))
        arr = (arr - self.mean) / self.std
        return torch.tensor(arr, dtype=torch.float32)

# TODO: Define train_transform and eval_transform based on above classes if helps. 
# Recommended train-time augmentations: rotation, crop, flip, color jitter.
# Recommended val/test preprocessing: resize + normalization only.
train_transform = None
eval_transform = None

def denormalize(t):
    t = t.clone().cpu()
    for c in range(3):
        t[c] = t[c] * 0.5 + 0.5
    return t.clamp(0, 1)

# TODO: Now you can use draw emoji function and test if it works well by some samples :

<style>
*{
    direction: rtl;
    text-align: right;
}
</style>

## بخش ۴ — ساخت Dataset و DataLoader

### صورت سؤال
1. یک کلاس `Dataset` بنویسید که با استفاده از رکوردهای هر split، تصویر ایموجی را بسازد.
2. برای train / val / test سه DataLoader ایجاد کنید.
3. یک batch را نمایش دهید و shape داده‌ها و برچسب‌ها را چاپ کنید.

### خروجی مورد انتظار
- کلاس دیتاست
- سه dataloader
- نمایش یک batch نمونه

In [ ]:
class EmojiMoodDataset(Dataset):
    """Build an image on the fly from a record.

    TODO inside __getitem__:
    - call draw_emoji using the record fields
    - apply transform if provided
    - return (image, label)
    """
    def __init__(self, records, transform=None):
        self.records = records
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        # TODO: generate the image for this record
        # TODO: apply transform if it exists
        # TODO: return the image tensor and the class index
        raise NotImplementedError('TODO: implement EmojiMoodDataset.__getitem__')

# TODO: instantiate datasets and dataloaders after the dataset class is implemented.
train_dataset = None
val_dataset = None
test_dataset = None
train_loader = None
val_loader = None
test_loader = None

In [ ]:
# TODO: once the dataloaders are ready, inspect one batch here.
# Example structure:
# images, labels = next(iter(train_loader))
# print('Batch image shape:', images.shape)
# print('Batch label shape:', labels.shape)
pass

<style>
*{
    direction: rtl;
    text-align: right;
}
</style>

## بخش ۵ — طراحی CNN پایه

### صورت سؤال
یک CNN پایه از صفر طراحی کنید که:
1. حداقل 3 بلوک کانولوشنی داشته باشد.
2. در انتها برای طبقه‌بندی از fully connected layers استفاده کند.
3. برای جلوگیری از overfitting از dropout استفاده کند.

### کارهای لازم
- مدل را پیاده‌سازی کنید.
- summary ساده‌ای از تعداد پارامترها گزارش کنید.
- یک forward pass آزمایشی انجام دهید.

### خروجی مورد انتظار
- تعریف مدل
- تعداد پارامترهای قابل آموزش
- shape خروجی مدل برای یک batch نمونه

In [ ]:
class BaselineCNN(nn.Module):
    """A small CNN baseline.

    TODO:
    - build at least 3 convolutional blocks
    - add a classifier head
    - implement forward
    """
    def __init__(self, num_classes=5):
        super().__init__()

        # TODO: complete the feature extractor.
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            # add more conv blocks here
        )

        # TODO: complete the classifier head.
        self.classifier = nn.Sequential(
            nn.Flatten(),
            # add linear + dropout layers here
        )

    def forward(self, x):
        # TODO: route x through the feature extractor and classifier
        raise NotImplementedError('TODO: implement BaselineCNN.forward')

# TODO: create the model and verify the output shape with a sample batch.
baseline_model = None

<style>
*{
    direction: rtl;
    text-align: right;
}
</style>

## بخش ۶ — توابع آموزش و ارزیابی

### صورت سؤال
توابع زیر را پیاده‌سازی کنید:
1. `train_one_epoch`
2. `evaluate_one_epoch`
3. یک حلقه‌ی آموزشی کامل که:
   - loss و accuracy را ذخیره کند
   - بهترین مدل را بر اساس validation loss نگه دارد

### نکته
در این تمرین از:
- `CrossEntropyLoss`
- بهینه‌ساز `Adam`

استفاده می‌کنیم.

### خروجی مورد انتظار
- توابع عمومی آموزش و ارزیابی

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch.

    TODO:
    - loop over batches
    - forward pass
    - compute loss
    - backward pass + optimizer step
    - collect predictions and targets
    - return average loss and accuracy
    """
    model.train()
    running_loss = 0.0
    all_preds, all_targets = [], []

    # TODO: implement the training loop body
    raise NotImplementedError('TODO: implement train_one_epoch')

@torch.no_grad()
def evaluate_one_epoch(model, loader, criterion, device):
    """Evaluate for one epoch.

    TODO:
    - loop over batches without gradients
    - compute average loss and accuracy
    """
    model.eval()
    running_loss = 0.0
    all_preds, all_targets = [], []

    # TODO: implement the evaluation loop body
    raise NotImplementedError('TODO: implement evaluate_one_epoch')

def fit_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    epochs=8,
    patience=3,
):
    """Train with validation tracking and early stopping.

    TODO:
    - store loss/accuracy history
    - keep the best model by validation loss
    - stop early if validation does not improve
    """
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': [],
    }
    # TODO: finish the training loop and return history
    raise NotImplementedError('TODO: implement fit_model')

<style>
*{
    direction: rtl;
    text-align: right;
}
</style>

## بخش ۷ — آموزش مدل پایه

### صورت سؤال
1. criterion و optimizer را تعریف کنید.
2. مدل پایه را برای چند epoch آموزش دهید.
3. تاریخچه‌ی آموزش را ذخیره کنید.
4. در پایان، بهترین نسخه‌ی مدل را نگه دارید.

### خروجی مورد انتظار
- لاگ آموزش در هر epoch

In [ ]:
# TODO: instantiate the baseline model, loss, and optimizer here.
# Example:
# baseline_model = BaselineCNN(num_classes=NUM_CLASSES).to(device)
# criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.Adam(baseline_model.parameters(), lr=1e-3, weight_decay=1e-4)
# baseline_history = fit_model(...)
pass

<style>
*{
    direction: rtl;
    text-align: right;
}
</style>

## بخش ۸ — رسم نمودارهای آموزش

### صورت سؤال
1. نمودار `train/val loss` را رسم کنید.
2. نمودار `train/val accuracy` را رسم کنید.
3. توضیح دهید آیا نشانه‌ای از overfitting یا underfitting مشاهده می‌شود یا خیر.

### خروجی مورد انتظار
- دو نمودار

In [ ]:
# TODO [11]: Complete the code for بخش ۸ — رسم نمودارهای آموزش.
# The instructions for this task are described in the markdown cell above.
# Replace this placeholder with your implementation.
# Hint: implement training/evaluation loops and track metrics.

<style>
*{
    direction: rtl;
    text-align: right;
}
</style>

## بخش ۹ — ارزیابی نهایی روی داده‌ی تست

### صورت سؤال
1. مدل را روی test set ارزیابی کنید.
2. معیارهای زیر را گزارش کنید:
   - Accuracy
   - Precision (macro)
   - Recall (macro)
   - F1-score (macro)
3. یک **Confusion Matrix** رسم کنید.
4. نتیجه را تفسیر کنید: کدام کلاس‌ها بیشتر با هم اشتباه می‌شوند؟

### خروجی مورد انتظار
- metricها
- confusion matrix
- classification report

In [ ]:
# TODO [12]: Complete the code for بخش ۹ — ارزیابی نهایی روی داده‌ی تست.
# The instructions for this task are described in the markdown cell above.
# Replace this placeholder with your implementation.
# Hint: implement training/evaluation loops and track metrics.

<style>
*{
    direction: rtl;
    text-align: right;
}
</style>

## بخش ۱۰ — تحلیل خطاها

### صورت سؤال
1. چند نمونه از تصاویر اشتباه‌طبقه‌بندی‌شده را نمایش دهید.
2. برای هر نمونه، برچسب واقعی و برچسب پیش‌بینی‌شده را بنویسید.
3. تلاش کنید برای خطاها توضیح بصری بدهید؛ مثلاً:
   - چرخش زیاد
   - نویز
   - شباهت بین mouth patternها
   - occlusion

### خروجی مورد انتظار
- چند تصویر misclassified
- تحلیل کیفی

In [ ]:
# TODO [13]: Complete the code for بخش ۱۰ — تحلیل خطاها.
# The instructions for this task are described in the markdown cell above.
# Replace this placeholder with your implementation.
# Hint: display misclassified samples or feature maps as requested.

<style>
*{
    direction: rtl;
    text-align: right;
}
</style>

## بخش ۱۱ — مشاهده‌ی feature mapها

### صورت سؤال
برای یکی از تصاویر تست:
1. خروجی اولین بلوک کانولوشنی را استخراج کنید.
2. چند feature map را نمایش دهید.
3. توضیح دهید که این feature mapها احتمالاً چه نوع الگوهایی را شکار می‌کنند.

### خروجی مورد انتظار
- تصویر ورودی
- چند feature map

In [ ]:
# TODO [14]: Complete the code for بخش ۱۱ — مشاهده‌ی feature mapها.
# The instructions for this task are described in the markdown cell above.
# Replace this placeholder with your implementation.


<style>
*{
    direction: rtl;
    text-align: right;
}
</style>

## بخش ۱۲ — مدل بهبود‌یافته

### صورت سؤال
اکنون یک مدل بهتر طراحی کنید که نسبت به مدل پایه، **پایدارتر** و **قابل‌تعمیم‌تر** باشد.

### پیشنهادها
- استفاده از `BatchNorm`
- استفاده از `AdaptiveAvgPool2d`
- استفاده از dropout بیشتر
- عمیق‌تر کردن مدل
- استفاده از optimizer و weight decay مناسب

### کارهای لازم
1. مدل جدید را پیاده‌سازی کنید.
2. آن را آموزش دهید.
3. عملکردش را با مدل پایه مقایسه کنید.

### خروجی مورد انتظار
- تعریف مدل جدید
- لاگ آموزش
- مقایسه‌ی metricها

In [ ]:
class BetterCNN(nn.Module):
    """A stronger CNN for the same task.

    TODO:
    - add BatchNorm and a deeper feature extractor
    - use AdaptiveAvgPool2d or a more robust head
    - implement forward
    """
    def __init__(self, num_classes=5):
        super().__init__()

        # TODO: build a more expressive feature extractor.
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            # add more blocks here
        )

        # TODO: build a stronger classifier head.
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(32, num_classes),
        )

    def forward(self, x):
        # TODO: pass x through features and head
        raise NotImplementedError('TODO: implement BetterCNN.forward')

# TODO: create the improved model.
better_model = None

In [ ]:
# TODO: train the improved model and compare it with the baseline.
# Example:
# criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.Adam(better_model.parameters(), lr=8e-4, weight_decay=1e-4)
# better_history = fit_model(...)
pass

In [ ]:
# TODO: after both models are trained, compare their test metrics here.
# Example outputs:
# - accuracy
# - precision_macro
# - recall_macro
# - f1_macro
# - parameter count
pass

<style>
*{
    direction: rtl;
    text-align: right;
}
</style>

## بخش ۱۳ — جمع‌بندی و پرسش‌های تحلیلی

### صورت سؤال
به پرسش‌های زیر پاسخ دهید:

1. آیا augmentation باعث بهتر شدن عملکرد مدل شد؟ چرا؟
2. کدام کلاس‌ها بیشتر با هم اشتباه شدند؟ دلیل بصری آن چیست؟
3. بین مدل پایه و مدل بهتر، کدام تغییرات بیشترین اثر را داشتند؟
4. آیا افزایش تعداد پارامترها همیشه به بهبود نتیجه منجر شد؟
5. اگر بخواهید این پروژه را سخت‌تر کنید، چه تغییری می‌دهید؟

### چند ایده برای بخش امتیازی
- ساخت یک **harder test set** با occlusion شدیدتر
- مقایسه‌ی optimizerهای `Adam` و `RMSprop`
- اضافه کردن **label smoothing**
- پیاده‌سازی **Grad-CAM** برای توضیح تصمیم مدل
- تبدیل مسئله از 5 کلاس به 7 کلاس با اضافه کردن `cool` و `confused`

